# Experiment 3: can the model learn tells of an approaching cold front?

**Question:** TiRex-2 can only use a past covariate if it discovers *from the history* that "this feature now
→ temperature change later". Can it do that for cold fronts? We give it a perfect tell (an artificial flag
12 h before every front), a fake tell (random flags), and the real precursors (pressure, humidity, wind).

**Cases:** real cold-front passages from the team labels (`fronts.csv`, `events.parquet`, PR #11): every accepted
front with enough history before it, 30 random points with a passage per front. Each forecast starts **6 h
before the temperature drop**, so the front arrives inside the 24 h forecast. History: **10 weeks**, so earlier
fronts are in it (Experiment 2: a longer history costs nothing extra).

| Setup | Model input besides temperature | Question |
|---|---|---|
| **A** | nothing | baseline |
| **B** | flag (past covariate): 1 at 12 h before every front passage of that point | can it learn "flag → front 12 h later"? |
| **C** | random flag (past covariate), same number of 1s at random hours | control: does any extra input change things? |
| **D** | pressure, humidity, wind (past covariates) | do the real precursors help? |
| **E** | front hours as a future covariate (1 during every drop, including the coming one) | does it use the flag when it is aligned with the drop (no time shift to learn)? |
| **F** | warning as a future covariate: 1 for the 12 h before every drop (also in the forecast, where it switches off as the drop starts) | does a longer, clearer signal work better than the 1-hour flag of B? |

**Measures**
- **drop captured (0-100 %)**: how much of the real temperature drop (largest fall within 6 h after the drop
  starts) the forecast also shows in that window. Only cases with a real drop of at least 2 °C.
- **error (°C)**: absolute difference between forecast (median) and what happened, averaged over 24 h.

In [1]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
from plotly.subplots import make_subplots
from tirex2 import TimeseriesType, load_model

DATA = Path("/workspace/data/austria_15km")
FIG = Path("/workspace/phase1-model-behavior/pair-1/experiments/figures/exp3")
FIG.mkdir(parents=True, exist_ok=True)
C, H = 10 * 168, 24  # history (10 weeks), forecast hours
BEFORE = 6           # forecast starts this many hours before the drop starts
FLAG_LEAD = 12       # setup B: flag this many hours before each drop
N_PER_FRONT = 30
QUICK = bool(os.environ.get("QUICK"))
if QUICK:  # quick code test
    N_PER_FRONT = 2
    FIG = Path("/tmp")  # test runs must not overwrite the real figures
RESULTS = Path(f"/workspace/data/notebooks/results/exp3_forecasts_{N_PER_FRONT}perfront.npz")
RERUN = False
SETUPS = {"A": "A: temperature only", "C": "C: random flag (control)", "B": "B: front flag 12 h ahead",
          "D": "D: pressure, humidity, wind", "E": "E: front hours given (future)",
          "F": "F: 12 h warning before drop (future)"}
COLOR = {"A": "#7f7f7f", "C": "#9467bd", "B": "#ff7f0e", "D": "#2ca02c", "E": "#1f77b4", "F": "#d62728"}

In [2]:
VARS = ["T2M", "P0", "RH2M", "UU", "VV"]
df = pd.read_parquet(DATA / "grid_long.parquet", columns=["point_id", "time", *VARS])
wide = {v: df.pivot(index="time", columns="point_id", values=v).interpolate(limit=3, limit_area="inside") for v in VARS}
times, pts = wide["T2M"].index, wide["T2M"].columns
col = {p: i for i, p in enumerate(pts)}
X = {v: w.to_numpy(np.float32) for v, w in wide.items()}
X["P0"] = X["P0"] / 100  # Pa -> hPa
T = X["T2M"]

fronts = pd.read_csv(DATA / "fronts.csv", parse_dates=["peak_time"])
fronts = fronts[fronts.cold_front].reset_index(drop=True)
ev = pd.read_parquet(DATA / "events.parquet")
onsets = (ev[ev.kind == "cold_front"].groupby("episode")
          .agg(point_id=("point_id", "first"), onset=("time", "min")).reset_index(drop=True))
print(f"{len(fronts)} accepted fronts, {len(onsets)} cold-front passages (all points)")

16 accepted fronts, 5672 cold-front passages (all points)


## 1. Timing check: label vs. actual drop

The label marks a passage when the 3 h cooling (as an anomaly from the usual cooling at that hour) is strong
enough. For each passage we look for the **steepest anomalous 3 h cooling within ±12 h** of the label and use
that as the anchor ("drop start"). The histogram shows how far the end of that drop is from the labelled hour
(0 = they agree).

In [3]:
d3 = np.full_like(T, np.nan)
d3[:-3] = T[3:] - T[:-3]  # temperature change over the next 3 h, starting at each hour
d3a = d3 - pd.DataFrame(d3).groupby([times.month, times.hour]).transform("mean").to_numpy()  # anomaly

rows = []
for _, f in fronts.iterrows():
    lo, hi = f.peak_time - pd.Timedelta("12h"), f.peak_time + pd.Timedelta("12h")
    on = onsets[(onsets.onset >= lo) & (onsets.onset < hi)].groupby("point_id").onset.min()
    for p, t in on.items():
        i = times.get_loc(t)
        w = np.arange(max(i - 12, 0), min(i + 13, len(times) - 3))
        j = int(w[np.nanargmin(d3a[w, col[p]])])
        rows.append({"front": f.peak_time, "point_id": p, "label": t, "drop_idx": j, "offset_h": j + 3 - i})
pas = pd.DataFrame(rows)

fig = go.Figure(go.Histogram(x=pas.offset_h, xbins={"start": -15.5, "end": 15.5, "size": 1}, marker_color="#7f7f7f"))
fig.update_layout(height=320, template="plotly_dark", bargap=0.05,
                  title=f"End of the actual drop minus labelled passage hour ({len(pas)} passages)",
                  xaxis={"title": "hours (negative = drop ends before the label)", "dtick": 3}, yaxis={"title": "passages"})
fig.write_html(FIG / "exp3_timing_check.html", include_plotlyjs="cdn")
fig.show()
print(f"label within ±1 h of the actual drop: {100 * (pas.offset_h.abs() <= 1).mean():.0f} % of passages, "
      f"within ±3 h: {100 * (pas.offset_h.abs() <= 3).mean():.0f} %")

label within ±1 h of the actual drop: 61 % of passages, within ±3 h: 89 %


## 2. Cases and inputs

In [4]:
drops = pas.groupby("point_id").drop_idx.apply(np.array).to_dict()  # all drop starts of each point
flag_b, flag_e, flag_f = np.zeros_like(T), np.zeros_like(T), np.zeros_like(T)
for p, js in drops.items():
    for j in js:
        if j - FLAG_LEAD >= 0:
            flag_b[j - FLAG_LEAD, col[p]] = 1
        flag_e[j : j + 3, col[p]] = 1
        flag_f[max(j - FLAG_LEAD, 0) : j, col[p]] = 1

rng = np.random.default_rng(0)
case_rows = []
for f, g in pas.groupby("front"):
    g = g[(g.drop_idx - BEFORE - C >= 0) & (g.drop_idx - BEFORE + H <= len(times))]
    if len(g) == 0:
        continue
    for _, r in g.iloc[rng.permutation(len(g))[:N_PER_FRONT]].iterrows():
        case_rows.append({"front": f, "point_id": r.point_id, "o": r.drop_idx - BEFORE})
cases = pd.DataFrame(case_rows)
if QUICK:
    cases = cases[cases.front.isin(cases.front.unique()[:2])].reset_index(drop=True)
truth = np.stack([T[o : o + H, col[p]] for p, o in zip(cases.point_id, cases.o)])
print(f"{cases.front.nunique()} fronts, {len(cases)} cases "
      f"(skipped fronts without {C // 168} weeks of history: {sorted(set(pas.front) - set(cases.front))})")


def zscore(x):
    return (x - np.nanmean(x, axis=1, keepdims=True)) / np.maximum(np.nanstd(x, axis=1, keepdims=True), 1e-3)


def series(setup, p, o, k):
    """Model input for one case."""
    c = col[p]
    target, past, fut = T[o - C : o, c][None], None, None
    if setup == "B":
        past = flag_b[o - C : o, c][None]
    elif setup == "C":
        n = max(int(flag_b[o - C : o, c].sum()), 1)
        past = np.zeros((1, C), np.float32)
        past[0, np.random.default_rng(k).choice(C, n, replace=False)] = 1
    elif setup == "D":
        past = zscore(np.stack([X[v][o - C : o, c] for v in ["P0", "RH2M", "UU", "VV"]]))
    elif setup == "E":
        fut = flag_e[o - C : o + H, c][None]
    elif setup == "F":
        fut = flag_f[o - C : o + H, c][None]
    t = lambda a: None if a is None else torch.from_numpy(np.ascontiguousarray(a, dtype=np.float32))
    return TimeseriesType(t(target), t(past), t(fut))

15 fronts, 450 cases (skipped fronts without 10 weeks of history: [Timestamp('2025-10-04 23:00:00+0000', tz='UTC')])


In [5]:
model = load_model("NX-AI/TiRex-2", device="cpu")
fc = {}
if RESULTS.exists() and not RERUN and not QUICK:
    z = np.load(RESULTS)
    assert list(z["point_id"]) == list(cases.point_id)
    fc = {s: z[s] for s in SETUPS if s in z.files}
    print(f"loaded saved forecasts from {RESULTS.name}: {', '.join(fc)}")
missing = [s for s in SETUPS if s not in fc]
if missing:
    for s in missing:
        t0 = time.time()
        ts = [series(s, p, o, k) for k, (p, o) in enumerate(zip(cases.point_id, cases.o))]
        out = model.forecast(ts, prediction_length=H, output_type="numpy", batch_size=32)
        fc[s] = np.stack([q[0] for q in out])
        print(f"{SETUPS[s]:32s} {time.time() - t0:5.0f} s")
    if not QUICK:
        RESULTS.parent.mkdir(exist_ok=True)
        np.savez_compressed(RESULTS, point_id=cases.point_id.to_numpy(), front=cases.front.astype(str).to_numpy().astype("U"),
                            start=np.asarray(times[cases.o].astype(str)).astype("U"), truth=truth, **fc)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

loaded saved forecasts from exp3_forecasts_30perfront.npz: A, C, B, D, E, F


## 3. Results

In [6]:
W = slice(BEFORE, BEFORE + 7)  # drop start and the 6 h after it
real_drop = truth[:, BEFORE] - truth[:, W].min(axis=1)
ok = real_drop >= 2


def captured(s):
    med = fc[s][:, 4]
    pred = med[:, BEFORE] - med[:, W].min(axis=1)
    return 100 * np.clip(pred / real_drop, 0, 1)


res = pd.DataFrame({"setup": [SETUPS[s] for s in SETUPS],
                    "drop captured, mean (%)": [captured(s)[ok].mean() for s in SETUPS],
                    "drop captured, median (%)": [np.median(captured(s)[ok]) for s in SETUPS],
                    "error (°C)": [np.abs(fc[s][:, 4] - truth).mean() for s in SETUPS]})
print(f"{ok.sum()} of {len(cases)} cases have a real drop of at least 2 °C (median drop {np.median(real_drop[ok]):.1f} °C)")
res.round(2)

269 of 450 cases have a real drop of at least 2 °C (median drop 6.0 °C)


/tmp/ipykernel_113/1671615694.py:9: RuntimeWarning: invalid value encountered in divide
  return 100 * np.clip(pred / real_drop, 0, 1)


,setup,"drop captured, mean (%)","drop captured, median (%)",error (°C)
0,A: temperature only,21.070000,5.28,3.87
1,C: random flag (control),21.170000,5.32,3.92
2,B: front flag 12 h ahead,21.059999,4.82,3.92
3,"D: pressure, humidity, wind",20.959999,4.80,3.93
4,E: front hours given (future),21.690001,4.75,4.01
5,F: 12 h warning before drop (future),21.650000,5.95,3.87


In [7]:
fig = make_subplots(rows=1, cols=2, column_widths=[0.65, 0.35], horizontal_spacing=0.1,
                    subplot_titles=["How much of the real drop does the forecast show?", "Error over the 24 h"])
for s, label in SETUPS.items():
    v = captured(s)[ok]
    fig.add_trace(go.Box(y=v, name=label, marker={"color": COLOR[s], "size": 3}, line={"color": COLOR[s]},
                         boxpoints="all", jitter=0.4, pointpos=0, showlegend=False,
                         hovertemplate="%{y:.0f} %<extra>" + label + "</extra>"), row=1, col=1)
    fig.add_trace(go.Bar(x=[label], y=[np.abs(fc[s][:, 4] - truth).mean()], marker_color=COLOR[s], showlegend=False,
                         hovertemplate="%{y:.2f} °C<extra>" + label + "</extra>"), row=1, col=2)
fig.update_yaxes(title_text="drop captured (%)", range=[-3, 103], row=1, col=1)
fig.update_yaxes(title_text="mean error (°C)", rangemode="tozero", row=1, col=2)
fig.update_xaxes(tickangle=-25)
fig.update_layout(height=520, template="plotly_dark",
                  title=f"Does the model use tells of an approaching cold front?<br><sup>{ok.sum()} front passages "
                        f"({cases.front.nunique()} fronts), forecast starts 6 h before the drop, 10 weeks of history. "
                        "One dot per case.</sup>")
fig.write_html(FIG / "exp3_drop_captured.html", include_plotlyjs="cdn")
fig.show()

/tmp/ipykernel_113/1671615694.py:9: RuntimeWarning: invalid value encountered in divide
  return 100 * np.clip(pred / real_drop, 0, 1)


## 4. The average front

All cases aligned on the start of the drop (hour 0). y = temperature change since the forecast started, averaged
over all cases. White = what really happened, colours = the mean forecast of each setup.

In [8]:
x = np.arange(H) - BEFORE
last = np.array([T[o - 1, col[p]] for p, o in zip(cases.point_id, cases.o)])[:, None]
fig = go.Figure()
fig.add_vrect(x0=0, x1=6, fillcolor="#1f77b4", opacity=0.07, line_width=0, annotation_text="drop window", annotation_position="top left")
fig.add_trace(go.Scatter(x=x, y=(truth - last).mean(axis=0), name="what happened", line={"color": "white", "width": 3.5}))
for s, label in SETUPS.items():
    fig.add_trace(go.Scatter(x=x, y=(fc[s][:, 4] - last).mean(axis=0), name=label, line={"color": COLOR[s], "width": 2.2},
                             hovertemplate="%{x:+d} h: %{y:+.1f} °C<extra>" + label + "</extra>"))
fig.update_layout(height=460, template="plotly_dark", legend={"orientation": "h", "y": -0.2},
                  title=f"The average cold front ({len(cases)} cases) and what each setup forecasts",
                  xaxis={"title": "hours from the start of the drop", "dtick": 3},
                  yaxis={"title": "change since forecast start (°C)"})
fig.write_html(FIG / "exp3_average_front.html", include_plotlyjs="cdn")
fig.show()

### Front by front

One panel per front (in date order), temperature only (setup A; the other setups look almost the same).
Thin grey lines = what happened at each point of that front, white = their mean, blue = the mean forecast.
Same change-since-forecast-start scale in every panel, so fronts of different strength can be compared.

In [9]:
fronts_sorted = sorted(cases.front.unique())
ncol = 5
nrow = int(np.ceil(len(fronts_sorted) / ncol))
fig = make_subplots(rows=nrow, cols=ncol, shared_xaxes=True, shared_yaxes=True, horizontal_spacing=0.02, vertical_spacing=0.08,
                    subplot_titles=[f"{pd.Timestamp(f):%d %b %Y}" for f in fronts_sorted])
for n, f in enumerate(fronts_sorted):
    r, c = n // ncol + 1, n % ncol + 1
    m = (cases.front == f).to_numpy()
    tr, fa = truth[m] - last[m], fc["A"][m, 4] - last[m]
    fig.add_vrect(x0=0, x1=6, fillcolor="#1f77b4", opacity=0.07, line_width=0, row=r, col=c)
    xs = np.r_[np.tile(np.r_[x, np.nan], m.sum())]
    fig.add_trace(go.Scatter(x=xs, y=np.c_[tr, np.full(m.sum(), np.nan)].ravel(), mode="lines",
                             line={"color": "rgba(120,120,120,0.25)", "width": 1}, hoverinfo="skip", showlegend=n == 0,
                             name="what happened at one point"), row=r, col=c)
    fig.add_trace(go.Scatter(x=x, y=tr.mean(0), mode="lines", line={"color": "white", "width": 3}, showlegend=n == 0,
                             name="what happened (mean)", hovertemplate="%{x:+d} h: %{y:+.1f} °C<extra>what happened</extra>"), row=r, col=c)
    fig.add_trace(go.Scatter(x=x, y=fa.mean(0), mode="lines", line={"color": "#1f77b4", "width": 3}, showlegend=n == 0,
                             name="forecast (mean)", hovertemplate="%{x:+d} h: %{y:+.1f} °C<extra>forecast</extra>"), row=r, col=c)
    fig.layout.annotations[n].text = f"{pd.Timestamp(f):%d %b %Y}<br><sup>{m.sum()} points, drop {np.median(real_drop[m]):.1f} °C</sup>"
fig.update_xaxes(dtick=6, range=[x[0], x[-1]])
fig.update_xaxes(title_text="h from drop start", row=nrow)
fig.update_yaxes(title_text="change (°C)", col=1)
fig.update_annotations(font={"size": 12})
fig.update_layout(height=260 * nrow + 120, template="plotly_dark", legend={"orientation": "h", "y": -0.08},
                  title=f"Every front, one panel: the forecast misses the drop each time<br><sup>{len(cases)} cases, {len(fronts_sorted)} fronts; "
                        "change since the forecast start, drop window shaded; median drop per front in each title</sup>",
                  margin={"t": 110})
fig.write_html(FIG / "exp3_fronts_small_multiples.html", include_plotlyjs="cdn")
fig.show()

## 5. One example case

The case with the median real drop (typical, not hand-picked). Top: the 10-week history; orange ticks = the
flag of setup B, red shading = the 12 h warnings of setup F, blue shading = the drops given to setup E (earlier fronts at this point). Bottom: the last
2 days and the forecasts of all setups.

In [10]:
cand = np.where(ok)[0] if ok.any() else np.arange(len(cases))
k = int(cand[np.argsort(real_drop[cand])[len(cand) // 2]])
p, o = cases.point_id[k], cases.o[k]
c = col[p]
fig = make_subplots(rows=2, cols=1, vertical_spacing=0.12, row_heights=[0.4, 0.6],
                    subplot_titles=["10-week history with the front flags", "Last 2 days and the forecasts"])
ht = times[o - C : o]
fig.add_trace(go.Scatter(x=ht, y=T[o - C : o, c], name="history", line={"color": "#7f7f7f", "width": 1}), row=1, col=1)
fb = np.where(flag_b[o - C : o, c] > 0)[0]
fig.add_trace(go.Scatter(x=ht[fb], y=T[o - C : o, c][fb], mode="markers", name="flag (setup B)",
                         marker={"color": COLOR["B"], "symbol": "triangle-down", "size": 11}), row=1, col=1)
for j in np.where(np.diff(np.r_[0, flag_f[o - C : o + H, c]]) > 0)[0]:
    t0 = times[o - C + j]
    for r in (1, 2):
        if r == 1 or t0 >= times[o - 60]:
            fig.add_vrect(x0=t0, x1=t0 + pd.Timedelta(f"{FLAG_LEAD}h"), fillcolor=COLOR["F"], opacity=0.08, line_width=0, row=r, col=1)
for j in np.where(np.diff(np.r_[0, flag_e[o - C : o + H, c]]) > 0)[0]:
    t0 = times[o - C + j]
    for r in (1, 2):
        if r == 1 or t0 >= times[o - 48]:
            fig.add_vrect(x0=t0, x1=t0 + pd.Timedelta("3h"), fillcolor=COLOR["E"], opacity=0.15, line_width=0, row=r, col=1)
ft = times[o : o + H]
fig.add_trace(go.Scatter(x=times[o - 48 : o], y=T[o - 48 : o, c], showlegend=False, line={"color": "#7f7f7f", "width": 1.5}), row=2, col=1)
fig.add_trace(go.Scatter(x=ft, y=truth[k], name="what happened", line={"color": "white", "width": 3}), row=2, col=1)
for s, label in SETUPS.items():
    fig.add_trace(go.Scatter(x=ft, y=fc[s][k, 4], name=label, line={"color": COLOR[s], "width": 2}), row=2, col=1)
fig.add_vline(x=ft[0], line={"color": "grey", "dash": "dot"}, row=2, col=1)
fig.update_yaxes(title_text="T2M (°C)")
fig.update_layout(height=700, template="plotly_dark", legend={"orientation": "h", "y": -0.12},
                  title=f"Point {p}, front of {cases.front[k]:%Y-%m-%d}, forecast from {times[o]:%Y-%m-%d %H:%M} UTC "
                        f"(real drop {real_drop[k]:.1f} °C)")
fig.write_html(FIG / "exp3_example_case.html", include_plotlyjs="cdn")
fig.show()

## 6. Forecasting from inside the front

So far every forecast started 6 h **before** the drop. Here the forecast starts later, so the model already sees
part or all of the drop in its history (temperature only, as setup A):

| Start | What the model has seen |
|---|---|
| −6 h | nothing of the front (setup A above) |
| 0 h | the hour the drop starts |
| +3 h | about half of the drop |
| +6 h | the whole drop (it usually takes 3-6 h) |
| +12 h | the drop and 6 h of the colder air behind it |

Question: once the front is in the history, does the model **keep the colder level**, or does it go back to
the warmer pattern of the weeks before? Same cases with a real drop of at least 2 °C as above.

In [11]:
OFFSETS = [-6, 0, 3, 6, 12]  # forecast start relative to the drop start (h)
RES_IN = RESULTS.with_name(RESULTS.name.replace("forecasts", "inside_front"))
j0 = cases.o.to_numpy() + BEFORE  # drop start of every case
sel = ok if ok.sum() >= 5 else np.ones(len(cases), bool)  # quick test may have no real drops
idx = np.where(sel & (j0 + max(OFFSETS) + H <= len(times)))[0]
REL = np.arange(-24, max(OFFSETS) + H)  # hours relative to the drop start
truth_w = np.stack([T[j0[k] + REL, col[cases.point_id[k]]] for k in idx])
fin = {}
if RES_IN.exists() and not RERUN and not QUICK:
    zi = np.load(RES_IN)
    assert list(zi["point_id"]) == list(cases.point_id[idx])
    fin = {off: zi[f"start{off:+d}"] for off in OFFSETS}
    print(f"loaded saved forecasts from {RES_IN.name}")
else:
    for off in OFFSETS:
        if off == -BEFORE:
            fin[off] = fc["A"][idx]
            continue
        t0 = time.time()
        ts = [TimeseriesType(torch.from_numpy(T[j0[k] + off - C : j0[k] + off, col[cases.point_id[k]]][None].copy()), None, None)
              for k in idx]
        out = model.forecast(ts, prediction_length=H, output_type="numpy", batch_size=64)
        fin[off] = np.stack([q[0] for q in out])
        print(f"start {off:+3d} h: {time.time() - t0:4.0f} s")
    if not QUICK:
        np.savez_compressed(RES_IN, point_id=cases.point_id[idx].to_numpy(), front=cases.front[idx].astype(str).to_numpy().astype("U"),
                            drop_start=np.asarray(times[j0[idx]].astype(str)).astype("U"), rel_hours=REL, truth=truth_w,
                            **{f"start{off:+d}": fin[off] for off in OFFSETS})


def at(off):
    """Truth for the 24 forecast hours of a start `off` h after the drop start."""
    i = np.searchsorted(REL, off)
    return truth_w[:, i : i + H]


ref = truth_w[:, np.searchsorted(REL, 0)][:, None]  # temperature when the drop starts
inside = pd.DataFrame({
    "forecast start": [f"{off:+d} h" for off in OFFSETS],
    "error (°C)": [np.abs(fin[off][:, 4] - at(off)).mean() for off in OFFSETS],
    "too warm (+) / too cold (−) (°C)": [(fin[off][:, 4] - at(off)).mean() for off in OFFSETS],
})
print(f"{len(idx)} cases (real drop ≥ 2 °C); for comparison, normal forecasts in Experiment 2 had an error of about 1.8 °C")
inside.round(2)

loaded saved forecasts from exp3_inside_front_30perfront.npz
269 cases (real drop ≥ 2 °C); for comparison, normal forecasts in Experiment 2 had an error of about 1.8 °C


,forecast start,error (°C),too warm (+) / too cold (−) (°C)
0,-6 h,4.09,3.53
1,+0 h,4.63,4.52
2,+3 h,2.72,1.93
3,+6 h,2.22,0.64
4,+12 h,2.25,0.51


### The average front, seen from five forecast starts

One panel per start. White = what really happened (average over all cases, change since the drop started), grey
area = what the model already knows, blue = the average forecast with its 10-90 % band.

In [12]:
SPAN = (-12, max(OFFSETS) + H)
fig = make_subplots(rows=1, cols=len(OFFSETS), shared_yaxes=True, horizontal_spacing=0.015,
                    subplot_titles=[f"start {off:+d} h" for off in OFFSETS])
m = (REL >= SPAN[0]) & (REL < SPAN[1])
for c, off in enumerate(OFFSETS, start=1):
    fig.add_vrect(x0=SPAN[0], x1=off, fillcolor="#7f7f7f", opacity=0.12, line_width=0, row=1, col=c)
    fig.add_vline(x=0, line={"color": "#bbbbbb", "dash": "dot", "width": 1}, row=1, col=c)
    fx = off + np.arange(H)
    lo, mid, hi = ((fin[off][:, q] - ref).mean(axis=0) for q in (0, 4, 8))
    fig.add_trace(go.Scatter(x=np.r_[fx, fx[::-1]], y=np.r_[hi, lo[::-1]], mode="lines", fill="toself", line={"width": 0},
                             fillcolor="rgba(31,119,180,0.18)", hoverinfo="skip", showlegend=c == 1, name="forecast band (10-90 %)"),
                  row=1, col=c)
    fig.add_trace(go.Scatter(x=REL[m], y=(truth_w - ref).mean(axis=0)[m], mode="lines", line={"color": "white", "width": 2.5},
                             name="what happened", showlegend=c == 1,
                             hovertemplate="%{x:+d} h: %{y:+.1f} °C<extra>what happened</extra>"), row=1, col=c)
    fig.add_trace(go.Scatter(x=fx, y=mid, mode="lines", line={"color": "#1f77b4", "width": 2.5}, name="forecast (median)",
                             showlegend=c == 1, hovertemplate="%{x:+d} h: %{y:+.1f} °C<extra>forecast</extra>"), row=1, col=c)
    fig.update_xaxes(range=SPAN, dtick=12, title_text="h from drop start" if c == 3 else None, row=1, col=c)
fig.update_yaxes(title_text="change since the drop started (°C)", row=1, col=1)
fig.update_layout(height=430, template="plotly_dark", legend={"orientation": "h", "y": -0.22},
                  title=f"Forecasting from inside the front ({len(idx)} cases)<br><sup>grey = what the model already knows; "
                        "dotted line = drop start</sup>")
fig.write_html(FIG / "exp3_inside_front.html", include_plotlyjs="cdn")
fig.show()

### Too warm or too cold, hour by hour

Forecast minus what happened, averaged over all cases, for each hour ahead. Above 0 = the forecast is too warm
(it expects the temperature to go back up), below 0 = too cold.

In [13]:
from plotly.colors import sample_colorscale

shades = sample_colorscale("Blues", [0.25 + 0.5 * i / (len(OFFSETS) - 1) for i in range(len(OFFSETS))])
fig = go.Figure()
fig.add_hline(y=0, line={"color": "white", "width": 1})
for off, colr in zip(OFFSETS, shades):
    fig.add_trace(go.Scatter(x=np.arange(1, H + 1), y=(fin[off][:, 4] - at(off)).mean(axis=0), mode="lines+markers",
                             line={"color": colr, "width": 2.5}, marker={"color": colr, "size": 5}, name=f"start {off:+d} h",
                             hovertemplate="%{x} h ahead: %{y:+.2f} °C<extra>" + f"start {off:+d} h" + "</extra>"))
fig.update_layout(height=430, template="plotly_dark",
                  title="Is the forecast too warm after the front?<br><sup>forecast minus what happened, mean over all cases; "
                        "lighter = earlier start</sup>",
                  xaxis={"title": "hours ahead", "dtick": 2}, yaxis={"title": "too warm (+) / too cold (−) (°C)"})
fig.write_html(FIG / "exp3_inside_front_bias.html", include_plotlyjs="cdn")
fig.show()

## 7. Every point of every front (for the dashboard)

Sections 3-6 use 30 random points per front. Here: **all** labelled points of every front, temperature only
(setup A), forecast starting 6 h before that point's drop, 10 weeks of history. The map shows, for one front at a
time (slider), how much of the real drop the forecast caught at each point. Grey = no real drop of at least 2 °C.

In [14]:
RES_ALL = RESULTS.with_name(RESULTS.name.replace(f"forecasts_{N_PER_FRONT}perfront", "all_points_A"))
allc = pas[(pas.drop_idx - BEFORE - C >= 0) & (pas.drop_idx - BEFORE + H <= len(times))].reset_index(drop=True)
if QUICK:
    allc = allc.iloc[:20]
allc["o"] = allc.drop_idx - BEFORE
truth_all = np.stack([T[o : o + H, col[p]] for p, o in zip(allc.point_id, allc.o)])
if RES_ALL.exists() and not RERUN and not QUICK:
    fc_all = np.load(RES_ALL)["forecast"]
    print(f"loaded saved forecasts from {RES_ALL.name}")
else:
    t0 = time.time()
    out = model.forecast([series("A", p, o, k) for k, (p, o) in enumerate(zip(allc.point_id, allc.o))],
                         prediction_length=H, output_type="numpy", batch_size=64)
    fc_all = np.stack([q[0] for q in out])
    print(f"{len(allc)} forecasts in {time.time() - t0:.0f} s")
    if not QUICK:
        np.savez_compressed(RES_ALL, point_id=allc.point_id.to_numpy(), front=allc.front.astype(str).to_numpy().astype("U"),
                            start=np.asarray(times[allc.o].astype(str)).astype("U"), truth=truth_all, forecast=fc_all)
# "unusual" = minus the usual temperature of that point, month and hour, so the normal evening cooling that the
# model always forecasts does not count as catching the front
usual_T = pd.DataFrame(T).groupby([times.month, times.hour]).transform("mean").to_numpy(np.float32)
us = np.stack([usual_T[o : o + H, col[p]] for p, o in zip(allc.point_id, allc.o)])


def drop_in(y):
    return y[:, BEFORE] - y[:, W].min(axis=1)


med = fc_all[:, 4]
rd, rd_u = drop_in(truth_all), drop_in(truth_all - us)
cap_raw = np.where(rd >= 2, 100 * np.clip(drop_in(med) / np.where(rd >= 2, rd, 1), 0, 1), np.nan)
cap = np.where(rd_u >= 2, 100 * np.clip(drop_in(med - us) / np.where(rd_u >= 2, rd_u, 1), 0, 1), np.nan)
allc["real_drop"], allc["unusual_drop"], allc["captured_raw"], allc["captured"] = rd, rd_u, cap_raw, cap
allc["drop_hour"] = times[allc.drop_idx].hour
print(f"{len(allc)} passages at {allc.point_id.nunique()} points, {allc.front.nunique()} fronts; "
      f"{(rd_u >= 2).sum()} with an unusual drop of at least 2 °C; unusual drop captured: mean {np.nanmean(cap):.0f} %, "
      f"median {np.nanmedian(cap):.0f} % (raw measure: mean {np.nanmean(cap_raw):.0f} %)")
allc.groupby("front").agg(points=("point_id", "size"), typical_drop_hour=("drop_hour", "median"),
                          unusual_drop_median=("unusual_drop", "median"), captured_raw_mean=("captured_raw", "mean"),
                          captured_unusual_mean=("captured", "mean"), cases_unusual=("captured", "count")).round(1)

loaded saved forecasts from exp3_all_points_A.npz
4724 passages at 378 points, 15 fronts; 4724 with an unusual drop of at least 2 °C; unusual drop captured: mean 10 %, median 6 % (raw measure: mean 21 %)


,points,typical_drop_hour,unusual_drop_median,captured_raw_mean,captured_unusual_mean,cases_unusual
front,,,,,,
2026-03-25 22:00:00+00:00,367,15.0,6.2,56.200001,16.1,367
2026-04-10 03:00:00+00:00,257,7.0,5.7,3.800000,4.5,257
2026-04-19 05:00:00+00:00,334,10.0,7.1,15.700000,8.7,334
2026-05-11 21:00:00+00:00,378,7.0,9.0,16.400000,13.9,378
2026-06-02 21:00:00+00:00,350,4.0,8.2,14.600000,5.0,350
2026-06-04 22:00:00+00:00,362,5.0,7.3,11.100000,4.1,362
2026-07-01 06:00:00+00:00,376,11.0,9.4,19.500000,5.6,376
2026-07-03 12:00:00+00:00,243,5.0,4.7,32.599998,5.3,243
2026-07-08 02:00:00+00:00,336,6.0,7.2,1.800000,10.3,336


In [15]:
P = pd.read_csv(DATA / "points_with_labels.csv").set_index("point_id")
flist = sorted(allc.front.unique())
fig = go.Figure()
fig.add_trace(go.Scattergeo(lon=P.lon, lat=P.lat, mode="markers", marker={"color": "#3a3a3a", "size": 5}, hoverinfo="skip", showlegend=False))
for i, f in enumerate(flist):
    g = allc[allc.front == f]
    has = g.captured.notna()
    fig.add_trace(go.Scattergeo(lon=P.lon.loc[g.point_id[~has]], lat=P.lat.loc[g.point_id[~has]], mode="markers", visible=i == 0, showlegend=False,
                                marker={"color": "#8a8a8a", "size": 7}, hovertemplate="no clear unusual drop<extra></extra>"))
    fig.add_trace(go.Scattergeo(lon=P.lon.loc[g.point_id[has]], lat=P.lat.loc[g.point_id[has]], mode="markers", visible=i == 0, showlegend=False,
                                marker={"color": g.captured[has], "cmin": 0, "cmax": 100, "colorscale": "Plasma", "size": 8,
                                            "colorbar": {"title": "drop<br>captured (%)"}},
                                text=[f"point {p}: unusual drop {d:.1f} °C, captured {c:.0f} %" for p, d, c in
                                      zip(g.point_id[has], g.unusual_drop[has], g.captured[has])],
                                hovertemplate="%{text}<extra></extra>"))
steps = []
for i, f in enumerate(flist):
    vis = [True] + [False] * (2 * len(flist))
    vis[1 + 2 * i], vis[2 + 2 * i] = True, True
    g = allc[allc.front == f]
    steps.append({"method": "update", "label": f"{pd.Timestamp(f):%d %b}",
                      "args": [{"visible": vis}, {"title.text": f"Front of {pd.Timestamp(f):%d %b %Y}: {len(g)} points, "
                                                              f"drop captured {g.captured.mean():.0f} % on average"}]})
g0 = allc[allc.front == flist[0]]
fig.update_geos(fitbounds="locations", showcountries=True, countrycolor="#666", showland=True, landcolor="#1e1e1e",
                bgcolor="rgba(0,0,0,0)", resolution=50)
fig.update_layout(height=560, template="plotly_dark", margin={"l": 0, "r": 0, "t": 60, "b": 0},
                  title=f"Front of {pd.Timestamp(flist[0]):%d %b %Y}: {len(g0)} points, drop captured {g0.captured.mean():.0f} % on average",
                  sliders=[{"active": 0, "steps": steps, "pad": {"t": 10}, "currentvalue": {"prefix": "front: "}}])
fig.write_html(FIG / "exp3_all_points_map.html", include_plotlyjs="cdn")
fig.show()

## Reusing the results

All forecasts are saved in `data/notebooks/results/exp3_forecasts_30perfront.npz`:

```python
z = np.load("/workspace/data/notebooks/results/exp3_forecasts_30perfront.npz")
z["point_id"], z["front"], z["start"]  # one entry per case (point, front peak time, forecast start, UTC)
z["truth"]                             # (cases, 24) what really happened, °C; the drop starts at hour index 6
z["A"], ..., z["F"]                    # (cases, 9 quantiles 10-90 %, 24 h) forecasts per setup
```

Section 7 (every point of every front, setup A) is in `exp3_all_points_A.npz`: `point_id`, `front`, `start`,
`truth` (passages, 24 h), `forecast` (passages, 9 quantiles, 24 h).

Section 6 (forecasts from inside the front) is in `exp3_inside_front_30perfront.npz`:

```python
z = np.load("/workspace/data/notebooks/results/exp3_inside_front_30perfront.npz")
z["point_id"], z["front"], z["drop_start"]  # one entry per case (only cases with a real drop of at least 2 °C)
z["rel_hours"], z["truth"]                  # truth from 24 h before to 36 h after the drop start, °C
z["start-6"], z["start+0"], z["start+3"], z["start+6"], z["start+12"]  # (cases, 9 quantiles, 24 h)
```